In [ ]:
# prepare_training.ipynb
# Stage 1 of 2: build + embed the review H5. Run this FIRST; then run
# training.ipynb (same pod). Splitting keeps data-prep and training separate so
# you can re-run training without rebuilding data.
# MULTI-VM: run this on ONE VM only -- it writes the shared embedding_h5.h5.
# Other VMs skip build/embed and just stage that H5 + train (see training.ipynb).
import os, tempfile

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"

# Log to LOCAL temp (per-VM, ephemeral), NOT /workspace: the network FS is shared,
# so a static /workspace log would have multiple VMs appending to one file
# (MooseFS cross-client O_APPEND is not atomic -> torn/interleaved lines). Durable
# state (H5, ledger, checkpoints) still lives on /workspace; the log is scratch.
LOG = os.path.join(tempfile.gettempdir(), "stable_query_latent", "pipeline.log")
os.makedirs(os.path.dirname(LOG), exist_ok=True)

print('repo:', REPO)
print('log :', LOG, '(local temp, per-VM)')

In [ ]:
# Clone or pull before every run.
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git remote -v
!git pull origin main
!git status --short
!git rev-parse HEAD


In [ ]:
# Start GPU + CPU + RAM + disk I/O monitor.
import sys, subprocess, threading, time, psutil
from pathlib import Path

if REPO not in sys.path:
    sys.path.insert(0, REPO)
try:
    # Same accounting the sweep planner budgets against (torch-free helper), so
    # what you see here == what the supervisor thinks is free.
    from VICReg_review.mem_budget import memory_breakdown as _mem_breakdown
except Exception:
    _mem_breakdown = None

stop = False

def _inline_breakdown():
    # Fallback if the repo isn't importable: read the cgroup directly.
    def _ri(p):
        try:
            t = Path(p).read_text().strip()
            return None if t == "max" else int(t)
        except Exception:
            return None
    limit = _ri("/sys/fs/cgroup/memory.max") or _ri("/sys/fs/cgroup/memory/memory.limit_in_bytes")
    stat = {}
    for p in ("/sys/fs/cgroup/memory.stat", "/sys/fs/cgroup/memory/memory.stat"):
        try:
            for line in Path(p).read_text().splitlines():
                k, _, v = line.partition(" ")
                if v.strip().isdigit():
                    stat[k] = int(v)
            if stat:
                break
        except Exception:
            pass
    if limit and stat and limit < 10**18:
        real = stat.get("anon", stat.get("rss", 0)) + stat.get("shmem", 0) + stat.get("unevictable", 0)
        cache = stat.get("file", stat.get("cache", 0))
        return {"source": "cgroup", "limit": float(limit), "real": float(real),
                "cache": float(cache), "usable": float(max(0, limit - real))}
    vm = psutil.virtual_memory()
    return {"source": "host", "limit": float(vm.total), "real": float(vm.total - vm.available),
            "cache": float(getattr(vm, "cached", 0)), "usable": float(vm.available)}

def get_memory_status():
    b = None
    if _mem_breakdown is not None:
        try:
            b = _mem_breakdown()
        except Exception:
            b = None
    if not b or not b.get("limit"):
        b = _inline_breakdown()
    lim = b["limit"] or 1.0
    return (b["real"] / lim * 100, b["real"] / 1024**3, b["cache"] / 1024**3,
            b["usable"] / 1024**3, b["limit"] / 1024**3, b["source"])

def get_gpu_status():
    # One entry per GPU.
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        cards = []
        for line in out.splitlines():
            p = [x.strip() for x in line.split(",")]
            cards.append(f"g{p[0]}:{float(p[1]):.0f}% {float(p[2])/1024:.1f}/{float(p[3])/1024:.1f}G")
        return " | ".join(cards) if cards else "n/a"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_real, ram_cache, ram_usable, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t
        print(
            f"[gpu] {gpu} | [cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] real {ram_real:.0f}/{ram_total:.0f} GiB "
            f"(usable {ram_usable:.0f}, cache {ram_cache:.0f}) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )
        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()

In [ ]:
# Build text H5.
!python -u game_review_data/build.py   --data-dir game_review_data/build_new_gamedata   --only metadata split text-h5   --split-device cuda   --logout-address {LOG}


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Embed text H5.
!python -u game_review_data/embedding_incloud.py   --input-h5 game_review_data/build_new_gamedata/text_h5.h5   --output-h5 game_review_data/embedding_h5.h5   --device cuda   --text-load-chunk-size 500000   --tok-workers 4   --logout-address {LOG}


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Stage the embedding H5 to FAST LOCAL disk (overlay/NVMe) if it fits; else fall
# back to the network volume (/workspace = MooseFS, ~256MB/s read cap). Only the
# big READ (training) is redirected via --h5; checkpoints/ledger keep writing to
# out_dir on /workspace (small I/O, the network is fine for that).
# Idempotent + non-fatal: any failure -> H5_PATH stays the /workspace H5.
import os, threading, time

WORKSPACE_H5 = os.path.join(REPO, "game_review_data/embedding_h5.h5")
LOCAL_H5 = "/root/data/embedding_h5.h5"
H5_PATH = WORKSPACE_H5   # safe default / fallback


def _free_bytes(path):
    st = os.statvfs(path)
    return st.f_bavail * st.f_frsize


def parallel_copy(src, dst, workers=8, chunk=64 << 20):
    """Thread-parallel byte-range copy (file I/O releases the GIL, so threads
    overlap the network reads; no fork/CUDA hazard from the notebook kernel)."""
    size = os.path.getsize(src)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "wb") as f:
        f.truncate(size)                       # preallocate
    bounds = [(i * size // workers, (i + 1) * size // workers) for i in range(workers)]
    errors = []

    def copy_range(start, end):
        sfd = os.open(src, os.O_RDONLY)
        dfd = os.open(dst, os.O_WRONLY)
        try:
            off = start
            while off < end:
                data = os.pread(sfd, min(chunk, end - off), off)
                if not data:
                    break
                os.pwrite(dfd, data, off)
                off += len(data)
        except BaseException as exc:            # noqa: BLE001
            errors.append(exc)
        finally:
            os.close(sfd)
            os.close(dfd)

    threads = [threading.Thread(target=copy_range, args=b) for b in bounds]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    if errors:
        raise errors[0]
    if os.path.getsize(dst) != size:
        raise IOError("size mismatch after copy")


try:
    src_size = os.path.getsize(WORKSPACE_H5)
    if os.path.exists(LOCAL_H5) and os.path.getsize(LOCAL_H5) == src_size:
        H5_PATH = LOCAL_H5
        print(f"local H5 already staged: {LOCAL_H5} ({src_size / 2**30:.1f} GiB)")
    else:
        free = _free_bytes("/")
        need = src_size + (5 << 30)            # 5 GiB headroom
        print(f"H5={src_size / 2**30:.1f}GiB  local('/') free={free / 2**30:.1f}GiB  need~{need / 2**30:.1f}GiB")
        if free >= need:
            t0 = time.time()
            parallel_copy(WORKSPACE_H5, LOCAL_H5, workers=8)
            dt = max(time.time() - t0, 1e-6)
            H5_PATH = LOCAL_H5
            print(f"staged -> {LOCAL_H5} in {dt:.0f}s ({src_size / 2**30 / dt:.2f} GiB/s)")
        else:
            print("not enough local space -> using /workspace H5 (expand Container Disk to stage locally)")
except BaseException as exc:
    print(f"staging failed ({type(exc).__name__}: {exc}) -> using /workspace H5")
    try:
        if os.path.exists(LOCAL_H5) and os.path.getsize(LOCAL_H5) != os.path.getsize(WORKSPACE_H5):
            os.remove(LOCAL_H5)
    except OSError:
        pass
    H5_PATH = WORKSPACE_H5

print("H5_PATH =", H5_PATH)


In [ ]:
# Smoke GATE: validate the TRAINING path on the freshly-built H5 with a tiny
# sweep (~4 combos x 2 epochs, separate cloud_smoke/ dir). Raises -> stops here
# if calibration / worker / train / ledger is broken, so data is only declared
# "ready" once the full pipeline is proven on it. Then training.ipynb is just
# the multi-hour run (no smoke needed there). Reads the staged local H5 (H5_PATH).
import json, shutil, subprocess
from collections import Counter
from pathlib import Path

# Fresh state each run: the ledger is append-only, so a previous smoke (e.g. a
# different train_game_counts) would leave stale entries that trip the all-done
# check below. cloud_smoke/ is throwaway, so wipe it first.
shutil.rmtree(Path(REPO, "VICReg_review/heads/cloud_smoke"), ignore_errors=True)

# calib.json (this GPU's memory model) + job queue -> machine-local scratch, not
# the shared FS; ledger/checkpoints stay under the smoke yaml's out_dir.
SMOKE_WORK = os.path.join(tempfile.gettempdir(), "stable_query_latent", "smoke_work")
os.makedirs(SMOKE_WORK, exist_ok=True)

subprocess.run(
    ["python", "-u", "VICReg_review/sweep/supervisor.py",
     "--config", "VICReg_review/sweep/sweep.smoke.yaml",
     "--h5", H5_PATH,
     "--work-dir", SMOKE_WORK,
     "--logout-address", LOG],
    cwd=REPO, check=True,
)

_smoke_ledger = Path(REPO, "VICReg_review/heads/cloud_smoke/ledger.jsonl")
_latest = {}
for _line in _smoke_ledger.read_text(encoding="utf-8").splitlines():
    _line = _line.strip()
    if not _line:
        continue
    try:
        _rec = json.loads(_line)
    except json.JSONDecodeError:
        continue
    if _rec.get("combo_id"):
        _latest[_rec["combo_id"]] = _rec
_counts = Counter(r.get("status", "?") for r in _latest.values())
print("smoke ledger:", dict(_counts))
assert _latest and _counts.get("done", 0) == len(_latest), f"SMOKE FAILED: {dict(_counts)} (fix before training.ipynb)"
print("smoke OK -> training pipeline validated on this H5")

In [ ]:
# Data prepared AND training pipeline smoke-validated. Final check on the H5.
from pathlib import Path
h5 = Path(REPO, 'game_review_data/embedding_h5.h5')
size_gb = (h5.stat().st_size / 1e9) if h5.exists() else 0.0
print(f'embedding_h5: {h5}\n  exists={h5.exists()}  size={size_gb:.2f} GB')
assert h5.exists(), 'embedding_h5.h5 missing -- re-run the build + embed cells'
print('Ready (data + smoke) -> run training.ipynb on this pod for the full sweep.')
